# Intro to Latent Variable Models and Pollux model classes

In [ ]:
# import warnings
# # daft draws its graphs with constrained layout, which matplotlib grumbles about
# warnings.filterwarnings("ignore", message="There are no gridspecs with layoutgrids")

import daft
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from helpers import make_simulated_linear_data

import pollux as plx
from pollux.models.transforms import LinearTransform

rng = np.random.default_rng(42)

Latent Variable Models (LVMs) are a very general class of models that are useful for many types of problems. For the purposes of Pollux and this note, LVMs are probabilistic models in which some of the variables are *latent* (i.e. unobserved) or noisily observed. More specifically within Pollux, we refer to vector-valued, per-object latent variables as "the latents" and all other (even latent) parameters are the "coefficients" or "parameters" of the model. This distinction might seem arbitrary, but the point is that the "latents" scale with the number of objects in the dataset whereas the "coefficients" do not. In this context, the latent variables are often used to capture (or discover) structure in the data and are sometimes interpretable. A general [probabilistic graphical model](https://adrian.pw/blog/probabilistic-graphical-models/) (PGM) for a latent variable of this form is shown below:

In [ ]:
pgm = daft.PGM(observed_style="shaded", dpi=150)
pgm.add_node("theta", r"$\theta$", 1, 3)
pgm.add_node("z", r"$z_n$", 2, 2)
pgm.add_node("y", r"$y_n$", 1, 1, observed=True)
for parent, child in [("z", "y"), ("theta", "y")]:
    pgm.add_edge(parent, child)
pgm.add_plate(
    [0.5, 0.5, 2.0, 2.0], label=r"$n = 1 \ldots N$", shift=0.0, position="bottom right"
)
pgm.render();

In the PGM above, $y_n$ are the observed data for object index $n$, $z_n$ are the latent variables for object index $n$, and $\theta$ are the coefficients of the model. (Remember that PGMs do not show the functional form of the model, only the conditional dependencies between the variables.) The plate indicates that there are $N$ objects in the dataset, each with its own latent variable and observed data. In general, $y$, $z$, and $\theta$ can all be vector valued. 

This is a very general model structure, and many models (or methods) you may be familiar with can be thought of as special cases of this general LVM structure. For example: 
- Principal component analysis (PCA) -- specifically probabilistic PCA --can be interpreted as an LVM in which the latents are the coefficients of the principal components and $\theta$ are the principal component vectors. (Or, more generally than pPCA, factor analysis is an LVM in which the latents are the coefficients of the factors and $\theta$ are the factors).
- Linear regression can be thought of as (degenerate form of) a LVM in which the latents are the true (noiseless) values of $y_n$ and the coefficients $\theta$ as the vector of linear parameters (e.g., slope and intercept). In this case, at fixed $\theta$ and covariates, each $z_n$ is determined rather than uncertain, so the per-object latents aren't actually free parameters.
- Gaussian mixture models (GMMs) are an example of an LVM where some latents are discrete: in a GMM, each $z_n$ is a discrete assignment of object $n$ to one of $K$ components, whereas the coefficients $\theta$ are the means, covariances, and mixing weights of those components.
- Various data-driven stellar spectroscopy models (e.g., The Cannon, Lux, etc.)are also LVMs. In this case, the data are the observed spectra of stars and the latents are often stellar labels (effective temperature, surface gravity, chemical abundances, etc.). The coefficients describe how the flux at each wavelength responds to those labels, and are shared across every star in the survey. 

At this point you might be wondering: are all probabilistic models LVMs? The answer is that many (or most?) generative models can be thought of as LVMs, but whether a model is usefully thought of as an LVM depends on the context. For example, a standard linear regression model is not useful to think of as an LVM, but one with uncertainties on both $x$ and $y$ and with missing or censored data might be more usefully thought of as an LVM.

TODO :draw a PGM and reference adrn blog post

Many problems or model structures that you might be familiar with can be expressed as LVMs. For example

I think of the per-object unknowns as the "latents" and otehrs as coefficients or parameters of functions.

Difference between linear/nonlinear

TODO


Are all probabilistic models LVMs? 



The other tutorials show you how to *use* Pollux. This one writes down the model, and
then checks every claim it makes with a few lines of code. It starts from what a latent
variable model is in general, narrows to the specific model this package implements, and
ends with the two things that most often trip people up: what the latent vectors do
*not* mean, and what a MAP fit does *not* give you.

If you want to get moving instead, start with
[Getting Started](LVM-getting-started.ipynb) and come back here when a result surprises
you.

## 1. The setup

We have $N$ objects. Each one is observed in many dimensions — say $D$ spectral pixels —
and we believe those observations are not really $D$-dimensional. Stars are made by a
handful of physical parameters, so a library of stellar spectra should lie close to a
surface of far lower dimension $L \ll D$ inside $\mathbb{R}^D$.

The oldest version of that idea is linear. Stack the spectra into a matrix
$Y \in \mathbb{R}^{N \times D}$ and look for a factorization

$$ Y \approx Z A^\top $$

where $Z \in \mathbb{R}^{N \times L}$ holds one short *latent vector* $z_n$ per object,
and $A \in \mathbb{R}^{D \times L}$ is shared by every object. Each spectrum is then a
weighted sum of $L$ basis spectra — the columns of $A$ — with $z_n$ supplying the
weights.

Let's simulate exactly such a dataset, with a true latent dimensionality of 4, and look
at its singular values.

(One housekeeping note: everywhere else in these tutorials you would attach a
preprocessor to rescale the data before fitting — see
[Getting Started](LVM-getting-started.ipynb). This page deliberately does not, so that
the arrays in the code map one-to-one onto the symbols in the equations. The simulated
data are already well scaled, so nothing is lost by skipping it here.)

In [ ]:
n_stars, n_latents, n_labels, n_flux = 512, 4, 2, 64

obs, truth = make_simulated_linear_data(
    n_stars=n_stars,
    n_latents=n_latents,
    n_labels=n_labels,
    n_flux=n_flux,
    rng=rng,
)

singular_values = np.linalg.svd(obs["flux"], compute_uv=False)

fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
ax.plot(np.arange(1, 13), singular_values[:12], marker="o", ls="none")
ax.axvline(n_latents + 0.5, color="tab:red", ls="--", lw=1)
ax.set(
    xlabel="component",
    ylabel="singular value",
    yscale="log",
    title=f"the data are really {n_latents}-dimensional",
)
_ = ax.text(
    n_latents + 0.8,
    singular_values[n_latents] * 1.7,
    "noise floor",
    color="tab:red",
    va="bottom",
)

The first four singular values stand well above the rest, and everything past them is
the noise we added. That cliff is the low-dimensional structure we are after, and for
data this clean an SVD finds it without any help.

## 2. Why an SVD is not enough

Real data break the SVD in four ways, and each one is a feature Pollux exists to
provide.

**Uncertainties differ from datum to datum.** An SVD minimizes unweighted squared error,
which is the right thing to do only if every pixel of every object has the same noise. A
spectrum with a bright sky line in one pixel, or a survey with a mix of exposure times,
badly violates that. We want each residual weighted by its own $1/\sigma^2$.

**Data go missing.** Not every object is measured in every dimension, and an SVD has no
notion of an absent entry.

**There is more than one kind of observation.** We do not just have spectra; we have
spectra *and* labels *and* photometry for the same stars, of different dimensions and in
different units. What ties them together is the belief that one latent vector per object
generates all of them. An SVD factors one matrix.

**The map need not be linear.** $Y \approx Z A^\top$ is a modeling choice, not a law.
The Cannon, for one, wants a quadratic.

Fixing the first two means writing down a likelihood instead of a matrix norm — which is
the step from PCA to *probabilistic* PCA and factor analysis. Fixing the last two means
letting that likelihood have several output blocks and an arbitrary map into each.
That's the model below.

## 3. The model Pollux fits

Every object gets a latent vector drawn from a prior. Each registered output $k$ is
generated from it by that output's transform, and observed with Gaussian noise:

$$
z_n \sim p(z), \qquad
y^{(k)}_n \sim \mathcal{N}\!\left(f_k(z_n;\, \theta_k),\ \Sigma^{(k)}_n\right)
$$

Here $f_k$ is the output's `data_transform` and $\theta_k$ are its parameters. The
covariance is diagonal and comes from the output's `err_transform` applied to the
reported uncertainties — by default that is the identity, so
$\Sigma^{(k)}_n = \mathrm{diag}\,(\sigma^{(k)}_n)^2$, but with a
{py:class}`~pollux.models.transforms.ScatterTransform` it becomes
$\sigma^2 + s^2$ for a fitted per-element $s$.

As a graph:

In [ ]:
pgm = daft.PGM(observed_style="shaded", dpi=110)
pgm.add_node("theta1", r"$\theta_1$", 1, 3.2)
pgm.add_node("theta2", r"$\theta_2$", 3, 3.2)
pgm.add_node("z", r"$z_n$", 2, 2)
pgm.add_node("y1", r"$y^{(1)}_n$", 1, 1, observed=True, aspect=1.3)
pgm.add_node("y2", r"$y^{(2)}_n$", 3, 1, observed=True, aspect=1.3)
for parent, child in [("z", "y1"), ("z", "y2"), ("theta1", "y1"), ("theta2", "y2")]:
    pgm.add_edge(parent, child)
pgm.add_plate([0.3, 0.3, 3.4, 2.4], label=r"$n = 1 \ldots N$", shift=-0.1)
pgm.render();

Shaded nodes are observed, open nodes are inferred, and the box is a *plate*: everything
inside it is repeated once per object. Note what sits where. The latents $z_n$ are inside
the plate — one per object. The transform parameters $\theta_k$ are outside it — one set,
shared by every object. That distinction drives most of the API, and we come back to it
in §4.

### The objective

{py:meth}`~pollux.models.LVM.optimize` runs stochastic variational inference with an
`AutoDelta` guide, which is a roundabout way of saying it finds the *maximum a
posteriori* parameters. It minimizes the negative log posterior over the latents and the
transform parameters jointly:

$$
-\ln p = \sum_{n}\sum_{k}\sum_{d}
\left[
\frac{\left(y^{(k)}_{nd} - f_k(z_n;\theta_k)_d\right)^2}{2\,\Sigma^{(k)}_{nd}}
+ \tfrac{1}{2}\ln \Sigma^{(k)}_{nd}
\right]
- \sum_n \ln p(z_n) - \sum_k \ln p(\theta_k)
$$

Three things to notice.

The first term is a $\chi^2$: this is a weighted fit, and that is the fix for
heteroscedastic errors. A datum with a huge uncertainty contributes almost nothing, which
is also how missing data are handled — give the absent entry an enormous error bar and it
drops out on its own. (That is exactly what the
[missing labels](LVM-hierarchical-missing-labels.ipynb) tutorial does.)

The second term, $\ln \Sigma$, does nothing at all when the uncertainties are fixed. It
matters the moment you fit a scatter $s$: without it, the optimizer would inflate $s$
until every residual was excused and the $\chi^2$ vanished. It is what keeps a fitted
error model honest.

The last two terms are the priors, which act as regularization. The prior on $\theta$ is
where you control how hard the transform parameters are shrunk toward zero.

Let's confirm the forward model is what it says on the tin — a two-output model whose
flux branch is a plain linear map.

In [ ]:
model = plx.LVM(latent_size=n_latents)
model.register_output("label", LinearTransform(output_size=n_labels))
model.register_output("flux", LinearTransform(output_size=n_flux))

# some arbitrary parameters to evaluate the forward model at
z = jnp.asarray(truth["latents"])
pars = {
    "latents": z,
    "label": {"data": {"A": jnp.asarray(truth["A"])}, "err": {}},
    "flux": {"data": {"A": jnp.asarray(truth["B"])}, "err": {}},
}

predicted = model.predict_outputs(pars)["flux"]
by_hand = z @ jnp.asarray(truth["B"]).T

print(
    "predict_outputs matches A z_n by hand:",
    bool(jnp.allclose(predicted, by_hand, atol=1e-4)),
)
print("shapes:", predicted.shape, "= (n_stars, n_flux)")

## 4. Two kinds of parameters

Look again at where things sit relative to the plate, because the split is not cosmetic:

- **Per-object parameters** live inside the plate. There are $N \times L$ latent numbers,
  and that count *grows with your dataset*. They describe the objects you happened to
  observe.
- **Shared parameters** live outside it. The size of $\theta_k$ is set by the model
  architecture, not by $N$. They describe the *relationship* between the latent space and
  the observable — and that is the part you can carry to objects you have not seen.

Pollux does not take your word for which is which. It measures it, by expanding each
transform's priors at two different dataset sizes and seeing which parameter shapes move:

In [ ]:
carried = model.output_pars(pars)

print(
    "per-object, tied to the stars we fitted:", sorted(model.per_object_param_names())
)
print(
    "shared, carried to new stars:          ", sorted(model.pack_numpyro_pars(carried))
)

### Train, then apply

That split is what makes label transfer possible. Fit everything on stars that have both
spectra and labels. Then, for a star with only a spectrum, hold the shared parameters
fixed and fit only its latent vector — and read the labels off the label branch, which
never saw a label for that star.

{py:meth}`~pollux.models.LVM.output_pars` builds the "carried over" half for you, and
`blocks=["latents"]` says to fit nothing else.

In [ ]:
train = plx.data.PolluxData(
    label=plx.data.OutputData(obs["label"][:256], err=obs["label_err"][:256]),
    flux=plx.data.OutputData(obs["flux"][:256], err=obs["flux_err"][:256]),
)
# the test stars have a spectrum and nothing else
test_flux_only = plx.data.PolluxData(
    flux=plx.data.OutputData(obs["flux"][256:], err=obs["flux_err"][256:])
)

trained = model.optimize_iterative(
    train, max_cycles=50, rng_key=jax.random.PRNGKey(0), progress=False
)
applied = model.optimize_iterative(
    test_flux_only,
    blocks=["latents"],
    fixed_pars=model.output_pars(trained.params),
    progress=False,
)

inferred = model.predict_outputs(applied.params, names="label")["label"]
rmse = np.sqrt(np.mean((inferred - obs["label"][256:]) ** 2, axis=0))
print("label RMSE from the spectrum alone:", np.round(rmse, 4))
print(
    "typical label uncertainty:        ",
    np.round(obs["label_err"][256:].mean(axis=0), 4),
)

## 5. Lux and the Cannon are the same equation

Pollux ships two named architectures. Neither is a different framework — each is a choice
of $f_k$ in the equation above.

{py:class}`~pollux.models.Lux` takes both outputs to be linear in the latents. The latent
vector is genuinely latent: it is inferred, and it is usually larger than the number of
labels and much smaller than the number of pixels.

In [ ]:
pgm = daft.PGM(observed_style="shaded", dpi=110)
pgm.add_node("A", r"$A$", 1, 3.2)
pgm.add_node("B", r"$B$", 3, 3.2)
pgm.add_node("s", r"$s$", 4.1, 3.2)
pgm.add_node("z", r"$z_n$", 2, 2)
pgm.add_node("label", r"$\ell_n$", 1, 1, observed=True)
pgm.add_node("flux", r"$f_n$", 3, 1, observed=True)
for parent, child in [
    ("z", "label"),
    ("z", "flux"),
    ("A", "label"),
    ("B", "flux"),
    ("s", "flux"),
]:
    pgm.add_edge(parent, child)
pgm.add_plate([0.3, 0.3, 3.4, 2.4], label=r"$n = 1 \ldots N$", shift=-0.1)
pgm.render();

{py:class}`~pollux.models.Cannon` makes a different choice: the latent vector *is* the
vector of stellar labels. There is no separate embedding to infer — the "latent" node is
measured, which in Pollux is expressed by registering the labels as an output with a
{py:class}`~pollux.models.transforms.NoOpTransform`. The spectrum is then a quadratic
function of the labels: a polynomial feature expansion followed by a linear map.

Notice what changes in the graph — the middle node is now shaded.

In [ ]:
pgm = daft.PGM(observed_style="shaded", dpi=110)
pgm.add_node("theta", r"$\theta$", 3, 3.2)
pgm.add_node("s", r"$s$", 4.1, 3.2)
pgm.add_node("label", r"$\ell_n$", 2, 2, observed=True)
pgm.add_node("flux", r"$f_n$", 3, 1, observed=True)
for parent, child in [("label", "flux"), ("theta", "flux"), ("s", "flux")]:
    pgm.add_edge(parent, child)
pgm.add_plate([0.9, 0.3, 2.8, 2.4], label=r"$n = 1 \ldots N$", shift=-0.1)
pgm.render();

That single difference in shading is the whole distinction between the two models, and it
explains their different failure modes. Because the Cannon's latents are pinned to
measured labels during training, its coefficient fit is a clean linear solve. Because
Lux's are free, it can use dimensions that no label describes — but it has more to
identify from the same data.

At test time the Cannon's label node is *un*shaded: inferring labels from a spectrum is
the same "hold $\theta$ fixed, fit the latents" move as in §4.

Both are ordinary `LVM` instances, and you can see the architecture in the transforms
they register:

In [ ]:
lux = plx.Lux(latent_size=16, label_size=n_labels, flux_size=n_flux)
cannon = plx.Cannon(label_size=n_labels, output_size=n_flux)

for name, m in [("Lux", lux), ("Cannon", cannon)]:
    print(f"{name} (latent_size={m.latent_size}):")
    for output, spec in m.outputs.items():
        transform = spec.data_transform
        inner = getattr(transform, "transforms", None)
        described = (
            " -> ".join(type(t).__name__ for t in inner)
            if inner
            else type(transform).__name__
        )
        print(f"    {output:6s}: {described}")

## 6. The latents are not identified

Here is the result that surprises people. Take any invertible $L \times L$ matrix $R$ and
rewrite the linear model:

$$ A z_n = \left(A R^{-1}\right)\left(R\, z_n\right) $$

Rotate every latent vector by $R$, rotate the matrix by $R^{-1}$, and *every prediction
is unchanged*. The likelihood cannot tell the two apart, because they are the same
function.

You might hope the prior breaks the tie. It does not: the default $\mathcal{N}(0, 1)$ on
the latents is isotropic, so it is invariant under rotations too. Its contribution
$\sum_n \|z_n\|^2$ is identical before and after.

In [ ]:
R = jnp.linalg.qr(jax.random.normal(jax.random.PRNGKey(3), (n_latents, n_latents)))[0]

rotated = {
    "latents": z @ R.T,
    "flux": {"data": {"A": pars["flux"]["data"]["A"] @ jnp.linalg.inv(R)}, "err": {}},
}

before = model.predict_outputs(pars, names="flux")["flux"]
after = model.predict_outputs(rotated, names="flux")["flux"]

print("predictions identical:  ", bool(jnp.allclose(before, after, atol=1e-4)))
print("prior term before/after:", float(jnp.sum(z**2)), float(jnp.sum((z @ R.T) ** 2)))

So a fit does not recover *the* latent basis; there is no such thing. What it does
recover is the **subspace** those basis vectors span. We can check that against the truth,
since we simulated the data ourselves: compare the column space of the fitted matrix with
the column space of the one that generated the fluxes, using the projection matrix onto
each (which, unlike the matrices themselves, is unique to the subspace).

In [ ]:
# The projection matrix onto the column space of M is unique to the subspace,
# unlike M itself, so it is the thing worth comparing.
def projector(M):
    Q, _ = jnp.linalg.qr(M)
    return Q @ Q.T


A_fit = trained.params["flux"]["data"]["A"]
B_true = jnp.asarray(truth["B"])
random_matrix = jax.random.normal(jax.random.PRNGKey(1), (n_flux, n_latents))

print("max |A_fit - B_true|, elementwise:", float(jnp.abs(A_fit - B_true).max()))
print()
print(
    "subspace distance ||P_fit - P_true||:",
    float(jnp.linalg.norm(projector(A_fit) - projector(B_true))),
)
print(
    "...for an unrelated random matrix:  ",
    float(jnp.linalg.norm(projector(random_matrix) - projector(B_true))),
)

The fitted matrix is nowhere near the true one entry by entry, and yet it spans
essentially the same subspace. Both statements are correct, and only the second one is
meaningful.

The practical consequence: **do not interpret individual latent dimensions.** "Latent 3
is the temperature direction" is not a claim the model supports — rerun the fit from a
different seed and latent 3 will be something else. Questions about the subspace as a
whole (how many dimensions the data support, what a direction *within* it predicts,
whether two stars are close in latent space) are well posed.

## 7. What a MAP fit gives you, and what it does not

`AutoDelta` returns the mode, not samples. There is no posterior width on $\theta$ at
all, and the per-object uncertainties you can get from
{py:meth}`~pollux.models.LVM.latent_uncertainties` are the curvature of the objective at
that mode:

$$ \mathrm{Cov}(z_n) \approx \left[ \nabla^2_{z_n} \left(-\ln p\right) \right]^{-1} $$

Because the output parameters are held fixed at their fitted values, these are
*conditional* on $\theta$ being right — an empirical-Bayes step. With a large training
set that is a mild assumption; with a small one it is not, and the error bars will be
too small.

In [ ]:
sigma_z = model.latent_uncertainties(test_flux_only, applied.params, names="flux")

print("shape:", sigma_z.shape, "= (n_test, latent_size)")
print("median per-dimension uncertainty:", np.round(np.median(sigma_z, axis=0), 4))

One more caveat worth stating plainly: this is the curvature *at a minimum*. For a model
that is linear in the latents the objective is exactly quadratic, so these are the exact
conditional widths. For a nonlinear one — the Cannon, say — it is a local approximation,
and away from a true minimum the Hessian can fail to be positive definite altogether.
Pollux returns `nan` there rather than a reassuring small number.

## Where to go next

- [Getting Started](LVM-getting-started.ipynb) — the same model, hands on.
- [Iterative optimization](LVM-iterative-optimization.ipynb) and
  [Linearized, closed-form solves](../linear-solves.md) — how the fit exploits the fact
  that this objective is quadratic in $z$ for fixed $\theta$, and in $\theta$ for fixed
  $z$, even though it is not jointly convex in both.
- [Error models](LVM-error-models.ipynb) — fitting $s$, and the $\ln \Sigma$ term from §3
  doing its job.
- [Missing labels](LVM-hierarchical-missing-labels.ipynb) — the "enormous error bar"
  trick for absent data.